# 03 — Qwen3.5-4B com currículo estruturado

Execução do experimento suplementar **Qwen3.5-4B few-shot** utilizando os currículos estruturados.

O notebook mantém a mesma organização do experimento principal com publicações: carregamento dos dados, inferência, avaliação semântica, métricas e resumo agregado.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [current, *current.parents] if (p / "src").exists()), current)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.parse_tags import parsear_tags_da_resposta, construir_ranking_qwen
from src.evaluation.semantic_matching import (
    SBERT_MODEL_NAME, carregar_qrels, construir_vocabulario, encodar_com_sbert,
    construir_matriz_similaridade, matching_greedy_1_to_1, salvar_npz,
    salvar_csv_avaliacoes_gerais,
)
from src.evaluation.metrics import (
    carregar_perfis_por_documento, avaliar_autor, salvar_csv_metricas,
    TOP_K_ALVO_METRICAS,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
CURRICULO_DIR = DATA_DIR / "curriculos"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "curriculum_tests" / "qwen3_5_4b"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CURRICULOS = CURRICULO_DIR / "curriculos_estruturados_padrao.json"
PERFIS_DOCUMENTO = CURRICULO_DIR / "perfis_documento_curriculo.json"
QRELS = PROCESSED_DIR / "ground_truth" / "LExR-prof-qrels_filtrado"

TAGS_BRUTAS = RESULTS_DIR / "tags_brutas_qwen3_5_4b_curriculo.json"
RANKINGS = RESULTS_DIR / "ranking_qwen3_5_4b_curriculo.json"
CHECKPOINT = RESULTS_DIR / "tags_brutas_qwen3_5_4b_curriculo_checkpoint.json"
CSV_METRICAS = RESULTS_DIR / "metricas_qwen3_5_4b_curriculo_por_autor.csv"
CSV_AVALIACOES = RESULTS_DIR / "avaliacoes_gerais_qwen3_5_4b_curriculo.csv"
SIM_DIR = RESULTS_DIR / "sim_matrices"
SIM_DIR.mkdir(parents=True, exist_ok=True)

for nome, caminho in {
    "Currículos estruturados": CURRICULOS,
    "Perfis por unidade curricular": PERFIS_DOCUMENTO,
    "Qrels": QRELS,
}.items():
    print(f"{nome:30s}: {'OK' if caminho.exists() else 'não encontrado'}")

## Configuração do modelo

Valores preservados do experimento original do Qwen3.5-4B: `do_sample=True`, `enable_thinking=False`, `temperature=0.7`, `top_p=0.8`, `top_k=20`, `min_p=0.0`, `repetition_penalty=1.0`, `max_new_tokens=1024`, `max_input_tokens=16384`, `batch_size=4` e `seed=42`.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
N_TAGS = 30
BATCH_SIZE = 4
DO_SAMPLE = True
ENABLE_THINKING = False
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0
REPETITION_PENALTY = 1.0
MAX_NEW_TOKENS = 1024
MAX_INPUT_TOKENS = 16384
SEED = 42
CHECKPOINT_EVERY = 12

SECOES_CURRICULO = [
    "formacao_academica",
    "atuacao_profissional",
    "areas_de_atuacao",
    "linhas_de_pesquisa",
    "ensino",
    "producao_bibliografica",
    "producao_tecnica",
    "orientacoes_concluidas",
    "demais_trabalhos",
]

## Carregamento dos currículos

In [ ]:
def normalizar_id_autor(valor):
    texto = str(valor).strip()
    return texto[3:] if texto.startswith("ID_") else texto


def carregar_autores_qrels(caminho):
    autores = set()
    with open(caminho, "r", encoding="utf-8") as f:
        for linha in f:
            partes = linha.rstrip("\n").split("\t")
            if partes and partes[0].strip():
                autores.add(partes[0].strip())
    return autores


autores_qrels = carregar_autores_qrels(QRELS)
mapa_ids = {normalizar_id_autor(a): a for a in autores_qrels}

with CURRICULOS.open("r", encoding="utf-8") as f:
    curriculos_raw = json.load(f)

curriculos = {}
for chave, dados in curriculos_raw.items():
    chave_norm = normalizar_id_autor(chave)
    if chave_norm not in mapa_ids or not isinstance(dados, dict):
        continue

    perfil = {
        secao: dados[secao]
        for secao in SECOES_CURRICULO
        if secao in dados and dados[secao] not in (None, "", [], {})
    }

    if perfil:
        curriculos[mapa_ids[chave_norm]] = perfil

print(f"Autores nos qrels: {len(autores_qrels):,}")
print(f"Currículos carregados: {len(curriculos):,}")

## Inferência few-shot

In [ ]:
SYSTEM_PROMPT = (
    "Você é um sistema especialista em perfilamento acadêmico e profissional "
    "e extração de perfis de expertise. Sua tarefa é analisar evidências "
    "curriculares de um autor e extrair suas principais áreas de expertise "
    "em forma de tags. Responda APENAS com um JSON contendo a lista de tags, "
    "sem texto adicional, sem markdown, sem explicações."
)

EXEMPLO_EN = {
    "formacao_academica": [{
        "tipo": "PHD",
        "curso": "Computer Science",
        "ano_conclusao": "2022",
        "titulo_trabalho": "Machine learning for clinical decision support"
    }],
    "atuacao_profissional": [{"vinculo_cargo": ["Researcher"]}],
    "areas_de_atuacao": ["Computer Science", "Artificial Intelligence", "Machine Learning"],
    "linhas_de_pesquisa": [{
        "linha": "Artificial intelligence in healthcare",
        "descricao": "Machine learning methods for clinical and biomedical data"
    }],
    "ensino": [{"disciplinas": ["Machine Learning", "Data Mining"]}],
    "producao_bibliografica": [
        {
            "tipo": "ARTICLES",
            "ano": "2024",
            "titulo": "Deep learning applied to image recognition in radiology",
            "palavras_chave": ["deep learning", "radiology", "computer vision"]
        },
        {
            "tipo": "ARTICLES",
            "ano": "2023",
            "titulo": "Machine learning approach for medical data mining",
            "palavras_chave": ["machine learning", "data mining", "medical informatics"]
        }
    ]
}

SAIDA_EN = {
    "tags": [
        "machine learning", "deep learning", "data mining", "medical informatics",
        "artificial intelligence", "medical imaging", "computer vision",
        "image recognition", "clinical decision support", "healthcare AI",
        "neural networks", "pattern recognition", "classification", "radiology",
        "biomedical data", "predictive modeling", "data analysis", "algorithms",
        "clinical informatics", "decision support"
    ]
}

EXEMPLO_PT = {
    "formacao_academica": [{
        "tipo": "DOUTORADO",
        "curso": "Química",
        "ano_conclusao": "2021",
        "titulo_trabalho": "Propriedades químicas de solos tropicais"
    }],
    "atuacao_profissional": [{"vinculo_cargo": ["Pesquisador"]}],
    "areas_de_atuacao": ["Ciência do Solo", "Química", "Agronomia"],
    "linhas_de_pesquisa": [{
        "linha": "Química e fertilidade de solos tropicais",
        "descricao": "Estudo de fertilidade, manejo e propriedades químicas do solo"
    }],
    "ensino": [{"disciplinas": ["Química Inorgânica", "Química do Solo"]}],
    "producao_bibliografica": [
        {
            "tipo": "ARTIGOS-PUBLICADOS",
            "ano": "2024",
            "titulo": "Química do solo em ambientes tropicais brasileiros",
            "palavras_chave": ["química do solo", "solos tropicais", "fertilidade"]
        },
        {
            "tipo": "ARTIGOS-PUBLICADOS",
            "ano": "2023",
            "titulo": "Ensino de química inorgânica no nível médio",
            "palavras_chave": ["ensino de química", "química inorgânica"]
        }
    ]
}

SAIDA_PT = {
    "tags": [
        "química do solo", "solos tropicais", "química inorgânica",
        "agronomia", "fertilidade do solo", "ensino de química",
        "manejo agrícola", "química ambiental", "ciência do solo",
        "educação química", "agricultura sustentável", "química aplicada",
        "propriedades químicas", "ensino médio", "didática",
        "solos brasileiros", "fertilidade", "química",
        "agronomia tropical", "manejo do solo"
    ]
}


def montar_mensagens(curriculo):
    instrucao = (
        f"Analise as informações curriculares a seguir e extraia as {N_TAGS} "
        "principais tags de expertise que melhor representam as áreas de conhecimento "
        "e atuação deste autor. Considere conjuntamente formação acadêmica, atuação "
        "profissional, áreas de atuação, linhas de pesquisa, ensino, produção "
        "bibliográfica, produção técnica, orientações e demais trabalhos, quando "
        "disponíveis. Cada tag deve ser curta (1 a 3 palavras), específica e "
        "descritiva. Ordene as tags da expertise mais representativa para a menos "
        "representativa. Não use nomes de pessoas, instituições ou cargos como tags. "
        "IMPORTANTE: mantenha o idioma predominante das evidências curriculares nas "
        "tags. Extraia as tags SOMENTE das evidências fornecidas. "
        'Responda APENAS com um JSON no formato: {"tags": ["tag1", "tag2", ...]} '
        "— sem markdown, sem comentários."
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "EXEMPLO 1 (apenas demonstração do formato esperado, NÃO copie estas tags):\n\n"
                       + instrucao + "\n\nCurrículo de exemplo:\n"
                       + json.dumps(EXEMPLO_EN, ensure_ascii=False, indent=2),
        },
        {"role": "assistant", "content": json.dumps(SAIDA_EN, ensure_ascii=False)},
        {
            "role": "user",
            "content": "EXEMPLO 2 (apenas demonstração do formato esperado, NÃO copie estas tags):\n\n"
                       + instrucao + "\n\nCurrículo de exemplo:\n"
                       + json.dumps(EXEMPLO_PT, ensure_ascii=False, indent=2),
        },
        {"role": "assistant", "content": json.dumps(SAIDA_PT, ensure_ascii=False)},
        {
            "role": "user",
            "content": "TAREFA REAL — analise o currículo REAL ABAIXO. NÃO repita as tags dos exemplos anteriores. "
                       "Extraia tags ESPECÍFICAS para este autor, baseadas SOMENTE nas evidências curriculares fornecidas.\n\n"
                       + instrucao + "\n\nCurrículo do autor:\n"
                       + json.dumps(curriculo, ensure_ascii=False, indent=2),
        },
    ]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)
model.eval()

tags_brutas = {}
if CHECKPOINT.exists():
    with CHECKPOINT.open("r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    tags_brutas = checkpoint.get("tags_brutas_llm", {})

autores = list(curriculos)

for inicio in tqdm(range(0, len(autores), BATCH_SIZE), desc="batches"):
    batch_autores = [a for a in autores[inicio:inicio+BATCH_SIZE] if a not in tags_brutas]
    if not batch_autores:
        continue

    prompts = [
        tokenizer.apply_chat_template(
            montar_mensagens(curriculos[a]),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
        for a in batch_autores
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            min_p=MIN_P,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
        )

    prompt_len = inputs.input_ids.shape[1]
    for i, autor in enumerate(batch_autores):
        resposta = tokenizer.decode(outputs[i][prompt_len:], skip_special_tokens=True).strip()
        tags_brutas[autor] = parsear_tags_da_resposta(resposta, n_tags_pedir=N_TAGS)

    if len(tags_brutas) % CHECKPOINT_EVERY < BATCH_SIZE:
        with CHECKPOINT.open("w", encoding="utf-8") as f:
            json.dump({"tags_brutas_llm": tags_brutas}, f, ensure_ascii=False)

with TAGS_BRUTAS.open("w", encoding="utf-8") as f:
    json.dump(tags_brutas, f, ensure_ascii=False)

rankings = {autor: construir_ranking_qwen(tags) for autor, tags in tags_brutas.items()}
with RANKINGS.open("w", encoding="utf-8") as f:
    json.dump(rankings, f, ensure_ascii=False)

print(f"Autores processados: {len(tags_brutas):,}")

## Avaliação semântica

SBERT `paraphrase-multilingual-mpnet-base-v2`, embeddings L2-normalizados, similaridade de cosseno, *global greedy* 1-para-1 e limiar 0,75.

In [ ]:
from sentence_transformers import SentenceTransformer

THRESHOLD_SBERT = 0.75
BATCH_SBERT = 256
SBERT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gt_norm, gt_original = carregar_qrels(QRELS)
perfis_doc = carregar_perfis_por_documento(PERFIS_DOCUMENTO)

vocab = construir_vocabulario(
    rankings=rankings,
    gt_norm=gt_norm,
    perfis_doc_semantico=perfis_doc,
    top_k_pred=TOP_K_ALVO_METRICAS,
)
print(f"Vocabulário SBERT: {len(vocab):,} strings")

modelo_sbert = SentenceTransformer(SBERT_MODEL_NAME, device=SBERT_DEVICE)
cache_emb = encodar_com_sbert(
    modelo_sbert,
    vocab,
    batch_size=BATCH_SBERT,
    device=SBERT_DEVICE,
)

if SBERT_DEVICE == "cuda":
    del modelo_sbert
    torch.cuda.empty_cache()

In [ ]:
dados_por_autor = {}
matching_por_autor = {}

for autor, ranking in rankings.items():
    if autor not in gt_norm:
        continue

    dados = construir_matriz_similaridade(
        autor=autor,
        ranking=ranking,
        gt_norm_autor=gt_norm[autor],
        cache_emb=cache_emb,
        top_k=TOP_K_ALVO_METRICAS,
    )
    if dados is None:
        continue

    matched_weights, matched_idx, matched_sims = matching_greedy_1_to_1(
        dados["sim"],
        dados["gold_weights"],
        theta=THRESHOLD_SBERT,
    )

    salvar_npz(
        dados,
        matched_idx,
        matched_weights,
        matched_sims,
        SIM_DIR,
    )

    dados_por_autor[autor] = dados
    matching_por_autor[autor] = {
        "matched_weights": matched_weights,
        "matched_idx": matched_idx,
        "matched_sims": matched_sims,
    }

print(f"Autores avaliados contra o gabarito: {len(dados_por_autor):,}")

## Métricas

In [ ]:
metricas_por_autor = {}

for autor, dados in dados_por_autor.items():
    metricas_por_autor[autor] = avaliar_autor(
        dados_autor=dados,
        matched_weights=matching_por_autor[autor]["matched_weights"],
        docs_ngrams_autor=perfis_doc.get(autor, {}),
        docs_campos_autor=perfis_doc.get(autor, {}),
        cache_emb=cache_emb,
        theta_cov=0.75,
    )

salvar_csv_metricas(
    metricas_por_autor,
    CSV_METRICAS,
    modelo="Qwen3.5-4B Currículo",
)

salvar_csv_avaliacoes_gerais(
    dados_por_autor=dados_por_autor,
    matching_por_autor=matching_por_autor,
    gt_original=gt_original,
    caminho=CSV_AVALIACOES,
    modelo="Qwen3.5-4B Currículo",
    rank_max=20,
)

print(f"Métricas: {CSV_METRICAS}")
print(f"Avaliação detalhada: {CSV_AVALIACOES}")

## Resumo agregado

In [ ]:
if metricas_por_autor:
    nomes = list(next(iter(metricas_por_autor.values())).keys())
    medias = {
        nome: float(np.mean([m[nome] for m in metricas_por_autor.values()]))
        for nome in nomes
    }
    for nome, valor in medias.items():
        print(f"{nome:20s}: {valor:.4f}")
else:
    print("Nenhuma métrica disponível.")

## Saídas

O notebook gera tags brutas, ranking posicional, matrizes `.npz`, métricas por autor e o CSV detalhado de pareamento.